In [10]:
import pandas as pd
import numpy as np 

In [11]:
df_raw = pd.read_csv("kidney_disease.csv")

#create copy
df = df_raw.copy()

#standardize column names
df.columns = df.columns.str.strip().str.lower()

#replace missing values with proper NaN
df = df.replace(["?", "", " "], pd.NA)

# strip whitespace from all object columns (gets rid of '\t', leading/trailing spaces)
obj_cols = df.select_dtypes("object").columns
df[obj_cols] = df[obj_cols].apply(lambda col: col.str.strip())

#fix messy category labels
df["dm"] = df["dm"].replace({"yes": "yes", "no": "no",
                             "\tyes": "yes", "\tno": "no", " yes": "yes"})
df["cad"] = df["cad"].replace({"no": "no", "yes": "yes", "\tno": "no"})
df["classification"] = df["classification"].replace({"ckd\t": "ckd"})

#convert numeric-like columns from strings to proper numeric
numeric_like = [
    "age", "bp", "sg", "al", "su",
    "bgr", "bu", "sc", "sod", "pot",
    "hemo", "pcv", "wc", "rc"
]
for col in numeric_like:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

#mark categorical columns and label them as type category
cat_cols = [
    "rbc", "pc", "pcc", "ba",
    "htn", "dm", "cad",
    "appet", "pe", "ane",
    "classification"
]
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")

#drop rows with any missing values to get a complete-case dataset
df_clean = df.dropna().reset_index(drop=True)

print("Original shape:", df_raw.shape)
print("Cleaned shape:", df_clean.shape)

#save cleaned data to new csv
df_clean.to_csv("kidney_disease_clean.csv", index=False)
print("Cleaned dataset saved as 'kidney_disease_clean.csv'")

Original shape: (400, 26)
Cleaned shape: (158, 26)
Cleaned dataset saved as 'kidney_disease_clean.csv'


In [12]:
#Create dummy variables for categorical columns (avoid multicollinearity with drop_first)
df_model = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)
df_model = df_model.drop(columns=["id","classification_notckd"])

print("Shape after dummy encoding:", df_model.shape)

# columns that should be 0/1 integers (if they exist)
bool_cols = [
    "rbc_normal", "pc_normal", "pcc_present", "ba_present",
    "htn_yes", "dm_yes", "cad_yes",
    "appet_poor", "pe_yes", "ane_yes",
    "classification_notckd"
]

for col in bool_cols:
    if col in df_model.columns:
        df_model[col] = df_model[col].astype(int)

# make everything numeric where possible
for col in df_model.columns:
    df_model[col] = pd.to_numeric(df_model[col], errors="ignore")

# save to final model ready csv
df_model.to_csv("kidney_disease_model_ready.csv", index=False)
print("Model-ready dataset saved as 'kidney_disease_model_ready.csv'")
print("\nFinal dtypes:")
print(df_model.dtypes)

Shape after dummy encoding: (158, 24)
Model-ready dataset saved as 'kidney_disease_model_ready.csv'

Final dtypes:
age            float64
bp             float64
sg             float64
al             float64
su             float64
bgr            float64
bu             float64
sc             float64
sod            float64
pot            float64
hemo           float64
pcv            float64
wc             float64
rc             float64
rbc_normal       int64
pc_normal        int64
pcc_present      int64
ba_present       int64
htn_yes          int64
dm_yes           int64
cad_yes          int64
appet_poor       int64
pe_yes           int64
ane_yes          int64
dtype: object


C:\Users\Koolt\AppData\Local\Temp\ipykernel_24256\1173639884.py:21: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_model[col] = pd.to_numeric(df_model[col], errors="ignore")


In [20]:
for i, col in enumerate(df.columns):
    col_type = "binary (0/1), discrete" if set(df[col].unique()) <= {0, 1} else "float, continuous"
    print(f"{i+1}. {col:20}  -->  {col_type}")


1. age                   -->  float, continuous
2. bp                    -->  float, continuous
3. sg                    -->  float, continuous
4. al                    -->  float, continuous
5. su                    -->  float, continuous
6. bgr                   -->  float, continuous
7. bu                    -->  float, continuous
8. sc                    -->  float, continuous
9. sod                   -->  float, continuous
10. pot                   -->  float, continuous
11. hemo                  -->  float, continuous
12. pcv                   -->  float, continuous
13. wc                    -->  float, continuous
14. rc                    -->  float, continuous
15. rbc_normal            -->  binary (0/1), discrete
16. pc_normal             -->  binary (0/1), discrete
17. pcc_present           -->  binary (0/1), discrete
18. ba_present            -->  binary (0/1), discrete
19. htn_yes               -->  binary (0/1), discrete
20. dm_yes                -->  binary (0/1), discrete